#STEP 1: Install Libraries

In [ ]:
import os

# Uninstall and reinstall bitsandbytes and accelerate for Colab compatibility
# This often helps resolve persistent ImportError issues
!pip install -U transformers datasets peft trl
!pip uninstall -y bitsandbytes accelerate

# Install latest versions of bitsandbytes and accelerate
# Check for GPU type for specific bitsandbytes installation if needed
# For general cases, a simple reinstall with -U should suffice
!pip install -U bitsandbytes accelerate

#STEP 2: Import Libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

#STEP 3: Load Dataset

In [ ]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")


In [ ]:
print(dataset[0])

#STEP 4: Convert to Instruction Format

In [ ]:
def format_instruction(example):
    return {
        "text": f"### Instruction:\n{example['Context']}\n\n### Response:\n{example['Response']}"
    }

dataset = dataset.map(format_instruction)

#STEP 5: Load Tokenizer

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#STEP 6: Tokenization

In [ ]:
def tokenize_fn(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize_fn)

#STEP 7: Load Model (8-bit for low RAM)

In [ ]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

#STEP 8: Apply LoRA (PEFT)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)

#STEP 9: Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-instruct",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

#STEP 10: Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

#STEP 11: Train

In [ ]:
trainer.train()

#STEP 12: Save LoRA Model

In [ ]:
model.save_pretrained("./tinyllama-instruct")
tokenizer.save_pretrained("./tinyllama-instruct")

#STEP 13: Load Model (Correct Way)

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./tinyllama-instruct")

#STEP 14: Inference (Chat Style)

In [ ]:
prompt = "### Instruction:\nI feel anxious and stressed.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(output[0], skip_special_tokens=True))